# Weekly CEDA EDA

This notebook analyzes the cleaned weekly state-level CEDA data. Each section includes a short interpretation below its plot.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..')
data_path = ROOT / 'data' / 'processed' / 'ceda_weekly_clean.csv'
df = pd.read_csv(data_path, parse_dates=['week'])
df = df.sort_values(['commodity', 'state', 'week'])
df.head()

In [ ]:
plt.figure(figsize=(12, 5))
for commodity, group in df.groupby('commodity'):
    trend = group.groupby('week', as_index=False)['modal_price'].median()
    plt.plot(trend['week'], trend['modal_price'], label=commodity)
plt.title('Weekly median price trends by commodity')
plt.xlabel('Week')
plt.ylabel('Rs/quintal')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Insight:** Commodity price levels show distinct seasonal and structural patterns.
Comparing these curves helps identify which commodities need separate model behavior.

In [ ]:
seasonal = df.assign(month=df['week'].dt.month).groupby(['commodity', 'month'], as_index=False)['modal_price'].median()
plt.figure(figsize=(12, 5))
sns.lineplot(data=seasonal, x='month', y='modal_price', hue='commodity', marker='o')
plt.title('Seasonality by calendar month')
plt.xlabel('Month')
plt.ylabel('Median Rs/quintal')
plt.tight_layout()
plt.show()

**Insight:** Monthly profiles reveal recurring seasonal demand and supply effects.
These patterns motivate calendar and festival-month features in the model.

In [ ]:
state_summary = df.groupby(['commodity', 'state'], as_index=False)['modal_price'].median()
plt.figure(figsize=(14, 6))
sns.barplot(data=state_summary, x='commodity', y='modal_price', hue='state')
plt.title('State comparison of median weekly prices')
plt.xlabel('Commodity')
plt.ylabel('Median Rs/quintal')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**Insight:** State medians differ by commodity, indicating geography matters even at state level.
State remains a categorical feature so the global model can learn these systematic differences.

In [ ]:
volatility = df.sort_values(['commodity', 'state', 'week']).copy()
volatility['weekly_return'] = volatility.groupby(['commodity', 'state'])['modal_price'].pct_change()
vol_summary = volatility.groupby('commodity', as_index=False)['weekly_return'].std().sort_values('weekly_return', ascending=False)
plt.figure(figsize=(12, 5))
sns.barplot(data=vol_summary, x='commodity', y='weekly_return', color='steelblue')
plt.title('Weekly price volatility by commodity')
plt.xlabel('Commodity')
plt.ylabel('Standard deviation of weekly return')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**Insight:** Volatility is uneven across commodities, so average error alone is not enough for model selection.
Directional accuracy and MASE provide complementary views of forecast quality.

In [ ]:
coverage = df.assign(present=1).pivot_table(index='commodity', columns='state', values='present', aggfunc='max', fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(coverage, annot=True, cmap='Blues', cbar=False)
plt.title('Series coverage availability')
plt.xlabel('State')
plt.ylabel('Commodity')
plt.tight_layout()
plt.show()

filled = df.groupby(['commodity', 'state'])['is_filled'].mean().unstack(fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(filled * 100, annot=True, fmt='.1f', cmap='Oranges')
plt.title('Filled-week share (%)')
plt.xlabel('State')
plt.ylabel('Commodity')
plt.tight_layout()
plt.show()

**Insight:** The coverage heatmap confirms which commodity/state series are available for modeling.
Filled-week shares show where interpolation contributes most and where forecasts need extra caution.